# Reproduce MACE - one click, in your browser

**Modular Algorithm Construction and Evolution for Combinatorial Optimization**

This notebook reproduces the paper's results with **no local installation**. Just press **Runtime > Run all** (or run the cells top to bottom).

- **Steps 1-2 need NO API key** - they reproduce the shipped heuristic portfolios.
- **Step 3 is optional** and runs the full MACE pipeline from scratch (needs your own OpenRouter key).
- **Step 4 is optional** and auto-generates a problem's I-O-T contract (output schema + tool library) from its description (needs your own OpenRouter key).

## Setup - clone the repository and install dependencies

The repository ships the exact 7109 benchmark instances used in the paper, so the clone is ~550 MB and takes 1-2 minutes.

In [ ]:
!git clone --depth 1 https://github.com/PJ-NTU/MACE.git
%cd MACE
!pip install -q -r requirements.txt
print("Setup done.")

## Step 1 - reproduce a shipped portfolio's scores (no API key)

Runs the paper's 10 evolved heuristics for a problem on its instances and reports the best-per-instance (complementary) score. Try any of the 40 problem folder names under `problems/`.

In [ ]:
!python scripts/evaluate_portfolio.py --problem aircraft_landing
print("--- another problem ---")
!python scripts/evaluate_portfolio.py --problem graph_colouring

## Step 2 - inspect everything without running code

All problem specifications, the seven operator prompt templates with rendered examples, the 400 final heuristic programs, and complete per-instance results are also browsable here:

**https://pj-ntu.github.io/MACE/mace_supplementary_information.html**

## Step 3 (optional) - run the full MACE pipeline from scratch

This generates a fresh heuristic portfolio with Stage One + Stage Two and **requires your own OpenRouter API key** (https://openrouter.ai/keys). A small demo run (`--N 3 --I-iter 1`) costs only a few cents and takes a few minutes.

Paste your key below and run the cell. The key stays in this private Colab session only.

In [ ]:
import os, getpass
os.environ['OPENROUTER_API_KEY'] = getpass.getpass('Paste your OpenRouter API key (input hidden): ')
!python scripts/run_mace.py --problem aircraft_landing --N 3 --I-iter 1 --T-max 6 --n-train 3 --n-test 2

## Step 4 (optional) - auto-generate a problem's I-O-T contract from its description

This runs the **contract generator** (`mace/contract/`): given a problem's
natural-language description, its **known** input loader (`load_data`), and a few
instances, an LLM automatically writes the **input schema (I)**, the **output
schema (O)**, and the **tool library T** (`is_feasible` + `objective`), plus a few
**domain helpers** - each admitted only after a real LLM-written heuristic solves
through the contract end to end.

Needs your OpenRouter key (reused from Step 3 if you ran it). A demo run costs a
few cents and takes a few minutes. Set `PROBLEM` to any folder under `problems/`;
the generated drop-in contract is written to `/tmp/<problem>_contract/`.

In [ ]:
# Step 4 (optional): auto-generate the I-O-T contract for a problem from its
# natural-language description, reusing the problem's known load_data.
# The LLM writes the input schema (I), the output schema (O), the tool library
# (is_feasible + objective), and a few domain helpers - each admitted only after
# a real LLM-written heuristic solves through the contract end to end.
import os, getpass
if not os.environ.get('OPENROUTER_API_KEY'):
    os.environ['OPENROUTER_API_KEY'] = getpass.getpass('Paste your OpenRouter API key (input hidden): ')

PROBLEM = 'aircraft_landing'   # change to any folder name under problems/
EXAMPLE = 'travelling_salesman_problem'   # a DIFFERENT problem used as a style example

# the problem's description = the natural-language problem statement
import importlib
cfg = importlib.import_module(f'problems.{PROBLEM}.config')
with open('/tmp/desc.txt', 'w') as f:
    f.write(cfg.DESCRIPTION)

!python scripts/generate_contract.py \
    --slug {PROBLEM} \
    --description /tmp/desc.txt \
    --instances problems/{PROBLEM}/instances \
    --load-data problems/{PROBLEM}/config.py \
    --example {EXAMPLE} \
    --model google/gemini-3.1-flash-lite \
    --out /tmp/{PROBLEM}_contract

print("\n===== auto-generated config.py (load_data + is_feasible + objective + helpers) =====\n")
print(open(f'/tmp/{PROBLEM}_contract/config.py').read()[:4000])